# Ch.2: Container Orchestration — Docker Compose

This notebook demonstrates how to orchestrate multi-container applications with Docker Compose.

**What you'll build:**
- Simple 2-service app (Flask + Redis)
- Production 3-tier stack (Flask + PostgreSQL + Redis)
- Health checks, persistent volumes, and dependency management

**Prerequisites:**
- Docker Desktop installed (includes Docker Compose)
- Basic understanding of Docker containers (see Ch.1)

## Cell 1: Verify Docker Compose Installation

Docker Compose comes bundled with Docker Desktop. Let's verify it's available.

In [ ]:
import subprocess
import sys

# Check Docker
result = subprocess.run(['docker', '--version'], capture_output=True, text=True)
if result.returncode != 0:
    print("❌ Docker not found. Install Docker Desktop first.")
    sys.exit(1)
print(f"✅ {result.stdout.strip()}")

# Check Docker Compose
result = subprocess.run(['docker', 'compose', 'version'], capture_output=True, text=True)
if result.returncode != 0:
    print("❌ Docker Compose not found. Upgrade Docker Desktop.")
    sys.exit(1)
print(f"✅ {result.stdout.strip()}")

print("\n✅ All prerequisites met!")

## Cell 2: Simple 2-Service Example (Flask + Redis)

Let's start with a minimal example: a Flask web app that counts page views using Redis.

**Architecture:**
```
Client → Flask (port 5000) → Redis (internal)
```

In [ ]:
import tempfile
from pathlib import Path
import os

# Create temporary working directory
work_dir = Path(tempfile.mkdtemp(prefix='compose_demo_'))
print(f"📁 Working directory: {work_dir}")

# Flask app with Redis counter
app_py = work_dir / 'app.py'
app_py.write_text('''
from flask import Flask
from redis import Redis

app = Flask(__name__)
redis = Redis(host='cache', port=6379, decode_responses=True)

@app.route('/')
def hello():
    count = redis.incr('hits')
    return f'Hello! This page has been viewed {count} times.\\n'

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=True)
'''.strip())

# Requirements
requirements = work_dir / 'requirements.txt'
requirements.write_text('flask==3.0.0\\nredis==5.0.1\\n')

# Dockerfile
dockerfile = work_dir / 'Dockerfile'
dockerfile.write_text('''
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
CMD ["python", "app.py"]
'''.strip())

# docker-compose.yml
compose_file = work_dir / 'docker-compose.yml'
compose_file.write_text('''
version: '3.9'

services:
  web:
    build: .
    ports:
      - "5000:5000"
    depends_on:
      - cache

  cache:
    image: redis:7-alpine
'''.strip())

print("✅ Created 2-service application:")
print(f"  - {app_py.name}")
print(f"  - {requirements.name}")
print(f"  - {dockerfile.name}")
print(f"  - {compose_file.name}")

## Cell 3: Start the 2-Service Stack

Now we'll use Docker Compose to build and start both services with one command.

In [ ]:
import subprocess
import time

os.chdir(work_dir)

print("🔨 Building and starting services...")
result = subprocess.run(
    ['docker', 'compose', 'up', '-d', '--build'],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(f"❌ Failed to start services:\\n{result.stderr}")
else:
    print("✅ Services started:")
    print(result.stdout)
    
    # Wait for services to be ready
    print("\\n⏳ Waiting for services to be healthy...")
    time.sleep(5)
    
    # Check running containers
    result = subprocess.run(
        ['docker', 'compose', 'ps'],
        capture_output=True,
        text=True
    )
    print("\\n📋 Running containers:")
    print(result.stdout)
    
    # Test the endpoint
    import requests
    try:
        response = requests.get('http://localhost:5000')
        print(f"\\n✅ Web service responding:")
        print(f"  Status: {response.status_code}")
        print(f"  Body: {response.text}")
        
        # Hit it a few more times to see counter increment
        for _ in range(3):
            response = requests.get('http://localhost:5000')
            print(f"  Body: {response.text}")
    except requests.exceptions.RequestException as e:
        print(f"❌ Could not reach web service: {e}")

## Cell 4: Add Database Service (PostgreSQL)

Now let's build a more realistic 3-tier stack: Flask API that stores users in PostgreSQL and caches queries in Redis.

We'll add:
- PostgreSQL service with persistent volume
- User CRUD endpoints in Flask
- Database connection with retry logic (handles startup race conditions)

In [ ]:
# Stop the current stack
print("🛑 Stopping current stack...")
subprocess.run(['docker', 'compose', 'down'], cwd=work_dir, capture_output=True)

# Create new 3-tier app
work_dir_v2 = Path(tempfile.mkdtemp(prefix='compose_3tier_'))
print(f"📁 New working directory: {work_dir_v2}")

# Updated Flask app with PostgreSQL
app_py_v2 = work_dir_v2 / 'app.py'
app_py_v2.write_text('''
import os
import time
from flask import Flask, jsonify, request
from redis import Redis
import psycopg2
from psycopg2.extras import RealDictCursor

app = Flask(__name__)

# Database connection with retry logic
def get_db_connection(retries=5):
    db_url = os.getenv('DATABASE_URL')
    for i in range(retries):
        try:
            conn = psycopg2.connect(db_url, cursor_factory=RealDictCursor)
            return conn
        except psycopg2.OperationalError as e:
            if i == retries - 1:
                raise
            print(f"DB not ready, retrying in 2s... ({e})")
            time.sleep(2)

# Initialize database
try:
    conn = get_db_connection()
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id SERIAL PRIMARY KEY,
                name VARCHAR(100) NOT NULL,
                email VARCHAR(100) UNIQUE NOT NULL
            )
        """)
    conn.commit()
    conn.close()
    print("✅ Database initialized")
except Exception as e:
    print(f"❌ Database init failed: {e}")

# Redis for caching
redis = Redis(host='cache', port=6379, decode_responses=True)

@app.route('/health')
def health():
    return jsonify({'status': 'ok'}), 200

@app.route('/users', methods=['GET'])
def get_users():
    # Check cache first
    cached = redis.get('users_list')
    if cached:
        redis.incr('cache_hits')
        return jsonify({'users': eval(cached), 'source': 'cache'})
    
    # Query database
    conn = get_db_connection()
    with conn.cursor() as cur:
        cur.execute("SELECT * FROM users")
        users = cur.fetchall()
    conn.close()
    
    # Cache result for 60 seconds
    redis.setex('users_list', 60, str(users))
    redis.incr('cache_misses')
    return jsonify({'users': users, 'source': 'database'})

@app.route('/users', methods=['POST'])
def create_user():
    data = request.json
    conn = get_db_connection()
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO users (name, email) VALUES (%s, %s) RETURNING *",
            (data['name'], data['email'])
        )
        user = cur.fetchone()
    conn.commit()
    conn.close()
    
    # Invalidate cache
    redis.delete('users_list')
    return jsonify({'user': user}), 201

@app.route('/stats')
def stats():
    hits = redis.get('cache_hits') or 0
    misses = redis.get('cache_misses') or 0
    return jsonify({
        'cache_hits': int(hits),
        'cache_misses': int(misses),
        'hit_rate': f"{int(hits) / (int(hits) + int(misses)) * 100:.1f}%" if int(hits) + int(misses) > 0 else "N/A"
    })

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=True)
'''.strip())

# Updated requirements
requirements_v2 = work_dir_v2 / 'requirements.txt'
requirements_v2.write_text('flask==3.0.0\\nredis==5.0.1\\npsycopg2-binary==2.9.9\\nrequests==2.31.0\\n')

# Dockerfile (same as before)
(work_dir_v2 / 'Dockerfile').write_text(dockerfile.read_text())

print("✅ Created 3-tier application files")

## Cell 5: Configure Environment Variables, Persistent Volumes, Health Checks

Now we'll write a production-ready `docker-compose.yml` with:
- Health checks for PostgreSQL
- Named volumes for database persistence
- Environment variables for configuration
- Proper startup ordering with `depends_on`

In [ ]:
# Production-ready docker-compose.yml
compose_v2 = work_dir_v2 / 'docker-compose.yml'
compose_v2.write_text('''
version: '3.9'

services:
  web:
    build: .
    ports:
      - "5000:5000"
    environment:
      - DATABASE_URL=postgresql://appuser:apppass@db:5432/appdb
    depends_on:
      db:
        condition: service_healthy
      cache:
        condition: service_started
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "python", "-c", "import requests; requests.get('http://localhost:5000/health')"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 40s

  db:
    image: postgres:16-alpine
    environment:
      POSTGRES_USER: appuser
      POSTGRES_PASSWORD: apppass
      POSTGRES_DB: appdb
    volumes:
      - postgres-data:/var/lib/postgresql/data
    restart: unless-stopped
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U appuser"]
      interval: 5s
      timeout: 3s
      retries: 5

  cache:
    image: redis:7-alpine
    restart: unless-stopped
    command: redis-server --appendonly yes
    volumes:
      - redis-data:/data

volumes:
  postgres-data:
  redis-data:
'''.strip())

print("✅ Created production-ready docker-compose.yml")
print("\\n🔑 Key features:")
print("  - Health checks ensure services are ready before starting dependents")
print("  - Named volumes persist data across restarts")
print("  - restart: unless-stopped ensures services recover from crashes")
print("  - Redis AOF persistence enabled")

## Cell 6: Start the 3-Tier Stack and Test CRUD Operations

Let's start all three services and test the complete user CRUD workflow.

In [ ]:
os.chdir(work_dir_v2)

print("🔨 Building and starting 3-tier stack...")
result = subprocess.run(
    ['docker', 'compose', 'up', '-d', '--build'],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(f"❌ Failed to start: {result.stderr}")
else:
    print("✅ Services started")
    
    # Wait for health checks
    print("\\n⏳ Waiting for services to be healthy...")
    time.sleep(10)
    
    # Check service status
    result = subprocess.run(['docker', 'compose', 'ps'], capture_output=True, text=True)
    print("\\n📋 Service status:")
    print(result.stdout)
    
    # Test CRUD operations
    import requests
    
    print("\\n🧪 Testing CRUD operations:")
    
    # Create users
    users = [
        {'name': 'Alice Johnson', 'email': 'alice@example.com'},
        {'name': 'Bob Smith', 'email': 'bob@example.com'},
        {'name': 'Carol Williams', 'email': 'carol@example.com'}
    ]
    
    for user in users:
        response = requests.post('http://localhost:5000/users', json=user)
        print(f"  Created: {response.json()['user']['name']} (status: {response.status_code})")
    
    # Query users (first time = cache miss)
    response = requests.get('http://localhost:5000/users')
    data = response.json()
    print(f"\\n  GET /users: {len(data['users'])} users found (source: {data['source']})")
    
    # Query again (should be cached)
    response = requests.get('http://localhost:5000/users')
    data = response.json()
    print(f"  GET /users: {len(data['users'])} users found (source: {data['source']})")
    
    # Cache stats
    response = requests.get('http://localhost:5000/stats')
    stats = response.json()
    print(f"\\n📊 Cache stats:")
    print(f"  Hits: {stats['cache_hits']}, Misses: {stats['cache_misses']}, Hit rate: {stats['hit_rate']}")

## Cell 7: Scale Services and View Logs

Docker Compose can scale services horizontally (multiple replicas). We'll also explore log aggregation across all services.

In [ ]:
# View logs from all services
print("📜 Recent logs from all services:")
result = subprocess.run(
    ['docker', 'compose', 'logs', '--tail=20'],
    capture_output=True,
    text=True,
    cwd=work_dir_v2
)
print(result.stdout)

# Note: Scaling the web service requires a load balancer in front
# (covered in Ch.7: Networking & Load Balancing)
print("\\n💡 Scaling web service:")
print("  docker compose up -d --scale web=3")
print("  ⚠️  Port conflict: All replicas try to bind to 5000")
print("  ✅ Solution: Use a reverse proxy (Nginx, Traefik) in front")

## Cell 8: Test Volume Persistence

Let's verify that database data survives container restarts by stopping all services, then starting them again.

In [ ]:
print("🛑 Stopping all services (preserving volumes)...")
subprocess.run(['docker', 'compose', 'down'], cwd=work_dir_v2, capture_output=True)

print("✅ Containers stopped")
print("\\n🔄 Starting services again...")
subprocess.run(['docker', 'compose', 'up', '-d'], cwd=work_dir_v2, capture_output=True)

time.sleep(10)

# Query users again - data should persist
import requests
response = requests.get('http://localhost:5000/users')
data = response.json()
print(f"\\n✅ Data persisted! {len(data['users'])} users found after restart")
for user in data['users']:
    print(f"  - {user['name']} ({user['email']})")

print("\\n💾 Volume persistence confirmed")
print("  - postgres-data volume survived 'docker compose down'")
print("  - To delete volumes: docker compose down --volumes")

## Cell 9: Cleanup

Stop all services and optionally remove volumes.

In [ ]:
print("🧹 Cleaning up...")

# Stop both demo directories
for dir_path in [work_dir, work_dir_v2]:
    if dir_path.exists():
        subprocess.run(['docker', 'compose', 'down'], cwd=dir_path, capture_output=True)
        print(f"  Stopped services in {dir_path.name}")

print("\\n✅ Cleanup complete")
print("\\n💡 To delete volumes (WARNING: deletes database data):")
print("  docker compose down --volumes")
print("\\n📊 List all volumes:")
print("  docker volume ls")

## Summary

**What you learned:**
1. ✅ Docker Compose orchestrates multi-container apps with one YAML file
2. ✅ `depends_on` with health checks ensures proper startup ordering
3. ✅ Named volumes persist data across container restarts
4. ✅ Internal DNS lets services communicate by name (e.g., `db`, `cache`)
5. ✅ Restart policies (`unless-stopped`) provide automatic recovery
6. ✅ Environment variables separate config from code

**Key commands:**
- `docker compose up -d` — Start all services in background
- `docker compose ps` — List running services
- `docker compose logs -f <service>` — Follow logs for a service
- `docker compose down` — Stop and remove containers (keeps volumes)
- `docker compose down --volumes` — Also delete volumes

**Next steps:**
- Run `notebook_supplement.ipynb` for Azure Container Instances deployment
- Move to Ch.3 for Kubernetes (multi-node orchestration)